In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()

In [ ]:
import os
from scipy import signal
import h5py
import numpy as np
import glob

VISUALIZATION_SAMPLE_SIZE = 50000
ANALYSIS_START_OFFSET = 20000
MAX_FILES_TO_PLOT = 5

NB_ID = "00"

In [ ]:
def load_signal_sample(filepath, num_samples, start_offset=0, key_filter='Gaussian'):
    """
    Robustly loads a sample slice from JamShield .mat files.
    Handles:
      - MATLAB v7.3 (HDF5) vs Legacy
      - Data Shape (2, N) -> [Real, Imag]
      - Missing keys (Fallbacks)
    """
    filepath = str(filepath)
    if not os.path.exists(filepath):
        print(f"Skipping missing file: {filepath}")
        return None

    try:
        # STRATEGY 1: HDF5 (Newer MATLAB v7.3)
        with h5py.File(filepath, 'r') as f:
            # Select Key (Prefer 'Gaussian' / 'Nojamming', else take first)
            available_keys = list(f.keys())
            target_key = key_filter if key_filter in f else available_keys[0]
            
            # Access Dataset
            dataset = f[target_key]
            total_cols = dataset.shape[1]

            start_idx = start_offset
            end_idx = min(start_offset + num_samples, total_cols)

            if start_idx >= total_cols:
                print(f"Warning: Offset {start_idx} is beyond file length {total_cols}")
                return None
            
            
            # Slice: Shape is (2, N) -> Row 0 is Real, Row 1 is Imag
            raw_slice = dataset[:, start_idx : end_idx]
            
            # Combine to Complex
            return raw_slice[0, :] + 1j * raw_slice[1, :]
                
    except Exception as e:
        print(f"Failed to load {os.path.basename(filepath)}: {e}")
        return None

In [ ]:
def load_all_scenarios(filepath, num_samples, start_offset):
    """
    Loads 'Nojamming' (Clean), 'Sine', and 'Gaussian' (Barrage) from one file.
    """
    # Map friendly names to HDF5 Keys
    scenarios = {
        'Clean Signal': 'Nojamming',
        'Sine Jamming': 'Sine',
        'Barrage (Gaussian)': 'Gaussian'
    }
    
    results = {}
    for label, key in scenarios.items():
        data = load_signal_sample(filepath, num_samples,start_offset, key_filter=key)
        if data is not None:
            results[label] = data
            
    return results

In [ ]:
def visualize_comparison(data_dict, filename, fs):
    """
    Generates a 3x3 Grid:
    Cols: Clean | Sine | Barrage
    Rows: Time | Spectrogram | Histogram
    """
    if not data_dict:
        print(f"No data found for {filename}")
        return

    labels = list(data_dict.keys())
    num_cols = len(labels)
    
    fig, axs = plt.subplots(3, num_cols, figsize=(20, 12), constrained_layout=True)
    fig.suptitle(f"Comparison: {filename} (Fs: {fs/1e6} MHz)", fontweight='bold')
    
    # Rows Definitions
    ROW_TIME = 0
    ROW_SPEC = 1
    ROW_HIST = 2

    for i, label in enumerate(labels):
        data = data_dict[label]
        
        # --- Row 1: Time Domain ---
        # Plot only first 1000 samples for clarity
        axs[ROW_TIME, i].plot(np.abs(data[:1000]), color='C0', lw=1)
        axs[ROW_TIME, i].set_title(f"{label}\nTime Domain (Zoomed)")
        if i == 0: axs[ROW_TIME, i].set_ylabel("Amplitude")

        # --- Row 2: Spectrogram ---
        f, t_spec, Sxx = signal.spectrogram(data, fs=fs, nperseg=256, return_onesided=False)
        f = np.fft.fftshift(f)
        Sxx = np.fft.fftshift(Sxx, axes=0)
        
        axs[ROW_SPEC, i].pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-12), 
                                    shading='gouraud', cmap='inferno')
        axs[ROW_SPEC, i].set_title("Spectrogram")
        if i == 0: axs[ROW_SPEC, i].set_ylabel("Frequency [Hz]")
        
        # --- Row 3: Histogram (I/Q Distribution) ---
        axs[ROW_HIST, i].hist(np.real(data), bins=80, alpha=0.6, label='Real', color='C0')
        axs[ROW_HIST, i].hist(np.imag(data), bins=80, alpha=0.6, label='Imag', color='C1')
        axs[ROW_HIST, i].set_title("I/Q Distribution", fontsize=12)
        if i == 0: 
            axs[ROW_HIST, i].set_ylabel("Count")
            axs[ROW_HIST, i].legend()
    
    slug = f"comparison_{filename.replace('.', '_')}"
    save_plot(slug, machine_id="A",nb_id=NB_ID, fig_id="01")
    plt.show()

In [ ]:
search_pattern = str(JAMSHIELD_RAW_DIR / JAMSHIELD_FILE_PATTERN)
files = sorted(glob.glob(search_pattern))

print(f"Found {len(files)} files. Visualizing first {MAX_FILES_TO_PLOT}...")
print(f"Sampling {VISUALIZATION_SAMPLE_SIZE} points per scenario.")
print("="*80)

for file_path in files[:MAX_FILES_TO_PLOT]:
    filename = os.path.basename(file_path)
    print(f"Processing {filename}...")
    
    # Load All 3 Versions
    scenarios_data = load_all_scenarios(
        file_path, 
        num_samples=VISUALIZATION_SAMPLE_SIZE,
        start_offset=ANALYSIS_START_OFFSET
    )
    
    # Visualize Comparison
    if scenarios_data:
        visualize_comparison(scenarios_data, filename, fs=JAMSHIELD_SAMPLE_RATE)
        print("-" * 100)
    else:
        print(f"Skipped {filename} (Could not extract scenarios)")